In [ ]:
from _init import *

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

In [ ]:
import random, math, glob
from transformers import TrainingArguments
from peft import LoraConfig, TaskType

from bait.utils import common_utils, json_utils, evaluation_utils
from bait.core import bait_utils
from bait.core.sft_trainer import SftTrainer

In [ ]:
seed = GlobalCommonConfig.SEED
common_utils.set_seed(seed)

In [ ]:
def load_datas(train_dir, train_file_name, train_size):
    train_file_path = f'{train_dir}/{train_file_name}'

    if train_file_path.endswith('.json'):
        all_datas = json_utils.load_json(train_file_path)
    elif train_file_path.endswith('.jsonl'):
        all_datas = json_utils.load_jsonl(train_file_path)

    random.shuffle(all_datas)

    train_datas = all_datas[:train_size]
    eval_datas = all_datas[train_size:]

    print(f'\n# sft_runner.load_datas() train_datas size : {len(train_datas)}')
    print(f'# sft_runner.load_datas() eval_datas size : {len(eval_datas)}\n')

    json_utils.write_jsonl(train_datas, f'{train_dir}/train_{len(train_datas)}.jsonl')
    json_utils.write_jsonl(eval_datas, f'{train_dir}/eval_{len(eval_datas)}.jsonl')

    return train_datas, eval_datas

In [ ]:
work_dir = f'/home/nlpshlee/dev_env/git/repos/bait'
data_dir = f'{work_dir}/data'
train_dir = f'{data_dir}/sft'

dtype = 'bfloat16'
device = 'cuda:0'
max_seq_length = 4096
max_new_tokens = 64

train_file_name = 'eval_1020.jsonl'
train_datas, eval_datas = load_datas(train_dir, train_file_name, 100)

In [ ]:
import copy
eval_datas = copy.deepcopy(train_datas)

In [ ]:
def get_peft_config(lora_r,
                    lora_alpha,
                    lora_dropout,
                    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
                    bias='none',
                    task_type=TaskType.CAUSAL_LM):
    
    peft_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        bias=bias,
        task_type=task_type
    )

    return peft_config

In [ ]:
def get_training_args(out_dir,
                      num_epochs,
                      batch_size,
                      accumulation_steps,
                      learning_rate,
                      weight_decay,
                      warmup_ratio,
                      max_grad_norm,
                      save_strategy='steps',
                      save_steps=10,
                      eval_strategy='steps',
                      eval_steps=10,
                      logging_steps=10,
                      lr_scheduler_type='cosine',
                      load_best_model_at_end=True,
                      metric_for_best_model='eval_loss',
                      save_total_limit=5):
    
    training_args = TrainingArguments(
        output_dir=out_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=accumulation_steps,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        warmup_ratio=warmup_ratio,
        max_grad_norm=max_grad_norm,

        save_strategy=save_strategy,
        save_steps=save_steps,
        eval_strategy=eval_strategy,
        eval_steps=eval_steps,
        logging_steps=logging_steps,

        lr_scheduler_type=lr_scheduler_type,
        load_best_model_at_end=load_best_model_at_end,
        metric_for_best_model=metric_for_best_model,
        save_total_limit=save_total_limit,

        bf16=True,
        do_train=True,
        label_names=['labels'],
        report_to='none'
    )

    return training_args

In [ ]:
def evaluate_all(checkpoint_dir, eval_datas):
    checkpoint_paths = glob.glob(f'{checkpoint_dir}/checkpoint-*')
    checkpoint_paths.sort(key=lambda x: int(x.split('-')[-1]))

    for checkpoint_path in checkpoint_paths:
        evaluation_utils.evaluate_sft(
            checkpoint_path,
            max_seq_length,
            max_new_tokens,
            eval_datas,
            device=device
        )

In [ ]:
model_names = ['Llama-3.2-3B', 'Llama-3.1-8B', 'Qwen2.5-3B', 'Qwen2.5-7B']

lora_r, lora_alpha, lora_dropout = 64, 128, 0.05
num_epochs, batch_size, accumulation_steps = 10, 1, 32
learning_rate, weight_decay, warmup_ratio, max_grad_norm = 5e-5, 0.01, 0.05, 1.0
save_and_eval_per_epoch, logging_per_epoch = 5, 5

real_batch_size = batch_size * accumulation_steps
steps_per_epoch = math.ceil(len(train_datas) / real_batch_size)
save_and_eval_steps = max(1, steps_per_epoch // save_and_eval_per_epoch)
logging_steps = max(1, steps_per_epoch // logging_per_epoch)

for model_name in model_names:
    model_name_or_path = bait_utils.get_model_name_or_path(model_name)

    sft_trainer = SftTrainer(model_name_or_path, max_seq_length, dtype, device)

    peft_config = get_peft_config(lora_r, lora_alpha, lora_dropout)
    sft_trainer.init_model(peft_config)

    out_dir = f'{train_dir}/trained/{model_name}'

    training_args = get_training_args(
        out_dir, num_epochs, batch_size, accumulation_steps,
        learning_rate, weight_decay, warmup_ratio, max_grad_norm,
        save_steps=save_and_eval_steps, eval_steps=save_and_eval_steps, logging_steps=logging_steps
    )

    sft_trainer.set_and_get_sft_dataset(train_datas, eval_datas)
    sft_trainer.train(training_args)
    sft_trainer.clear()

In [ ]:
model_names = ['Llama-3.2-3B', 'Llama-3.1-8B', 'Qwen2.5-3B', 'Qwen2.5-7B']

for model_name in model_names:
    model_name_or_path = bait_utils.get_model_name_or_path(model_name)

    out_dir = f'{train_dir}/trained/{model_name}'

    evaluate_all(out_dir, eval_datas)